In [3]:
# 2. Code Generation with ReACT Prompting

#Goal: Generate Python code using reasoning before execution.

import os
from dataclasses import dataclass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
_HAS_OPENAI = False
try:
    if OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI()
        _HAS_OPENAI = True
except Exception:
    _HAS_OPENAI = False

@dataclass
class LLMResponse:
    text: str

class MockLLM:
    def chat(self, prompt: str) -> LLMResponse:
        p = prompt.lower()
        if "generate python code" in p:
            code = (
                "def unique_sorted_squares(nums):\n"
                "    return sorted({n*n for n in nums})\n"
                "\n"
                "print(unique_sorted_squares([3,-1,2,2,-3,0]))"
            )
            return LLMResponse(code)
        if "summarize" in p and "reflect" not in p:
            return LLMResponse("Summary: LLMs can hallucinate; clear benchmarks and tool use reduce risk.")
        if "critique" in p or "reflect" in p:
            return LLMResponse("Critique: Add one concrete example and an evaluation best practice.")
        if "improved summary" in p:
            return LLMResponse("Improved Summary: LLMs may hallucinate, but using retrieval, human review, and clear benchmarks mitigates risks.")
        return LLMResponse("OK")

class OpenAILLM:
    def chat(self, prompt: str) -> LLMResponse:
        resp = client.responses.create(
            model="gpt-4o-mini",
            input=[{"role":"user","content":prompt}],
            temperature=0.3,
        )
        out = []
        for o in getattr(resp, "output", []):
            if getattr(o, "type", "") == "output_text":
                out.append(o.text)
        return LLMResponse("\n".join(out) if out else str(resp))

LLM = OpenAILLM() if _HAS_OPENAI else MockLLM()
print("Using OpenAI:", _HAS_OPENAI)


Using OpenAI: False


In [4]:

import re, traceback

def react_generate_code(task_description: str) -> str:
    prompt = f'''You are a Python coding assistant using a ReACT style.
    First, briefly describe your plan prefixed with "Plan:".
    Then, output only a Python solution in a fenced code block.

    Task: {task_description}
    '''
    return LLM.chat(prompt).text

def extract_code_block(text: str):
    m = re.search(r"```python\n(.*?)\n```", text, flags=re.DOTALL)
    return m.group(1) if m else text

def try_execute(py_code: str):
    try:
        local_vars = {}
        exec(py_code, {}, local_vars)
        return True, local_vars, ""
    except Exception as e:
        import traceback
        return False, {}, traceback.format_exc()

# Example
task = "Generate Python code that defines a function unique_sorted_squares(nums) and prints the result for [3, -1, 2, 2, -3, 0]."
result = react_generate_code(task)
print(result)
code = extract_code_block(result)
ok, ns, err = try_execute(code)
print("\n--- Execution Result ---")
print("Success ✅" if ok else f"Error:\n{err}")


def unique_sorted_squares(nums):
    return sorted({n*n for n in nums})

print(unique_sorted_squares([3,-1,2,2,-3,0]))
[0, 1, 4, 9]

--- Execution Result ---
Success ✅
